In [1]:
import torch 
import torch.nn as nn
import torch.optim as optim 

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
# DATASETS AND dataloader 

from torch.utils.data import DataLoader 
import torchvision.transforms as transforms

# img => scale(0,1) => normalize = (-1,1)

transform = transforms.Compose ([ #compose - chain all transformation
    transforms.ToTensor(), #convert it in pytorch tensor and scale 
    transforms.Normalize((0.5 , 0.5 , 0.5 ) , (0.5 , 0.5 , 0.5)) #normalize(std dev , mean)
])

# cifar already have train and test set
trainset = CIFAR10(root="./data" , train=True , download=True , transform = transform) #download - if not exists the  only
testset = CIFAR10(root="./data" , train=False , download=True , transform = transform) #download - if not exists the  only

In [3]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [4]:
trainloader = DataLoader(trainset , batch_size = 64 , shuffle=True)
testloader = DataLoader(testset , batch_size = 64)

## Build CNN

In [ ]:
class CNN (nn.Module):
    def __init__(self):
        super(CNN , self).__init__()

        self.conv_layers = nn.Sequential(
            # 1st layer
            # 2d - 2d matrix - image (32,32,3)
            nn.Conv2d(3 , 32 , kernel_size =3, padding=1 ), # input channel =3, output channel/feature maps , kernel=3x3
            nn.ReLU(),
                # output = (32,32,32) (width , height , channel)
            nn.MaxPool2d(2,2), #kernel size =2 , stride = 2
                 # output = (16,16,32)
 
            # 2nd layer
            nn.Conv2d(32 , 64 , kernel_size =3, padding=1 ) ,
            nn.ReLU(),
                # output = (16,16,64)
            nn.MaxPool2d(2,2),
               # output = (8,8,64)
 
            # 3rd layer
            nn.Conv2d(64 , 128 , kernel_size =3, padding=1 ) ,
            nn.ReLU(),
                # output = (8,8,128)
            nn.MaxPool2d(2,2) ,
                # output = (4,4,128) 
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128 , 256), #input , output
            nn.ReLU(),

                # output -- softmax applied byt cross entropy
            nn.Linear(256, 10) ,#10 classes
        )

    def forward(self , x):
        x =  self.conv_layers(x)
        x = x.view(x.size(0) , -1) #flattening x
        x = self.fc_layers(x)

        return x
        

In [6]:
model = CNN()

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

### Training the cnn


In [8]:
epochs = 10 
for epoch in range(epochs) :
    epoch_training_loss = 0.0 

    for images , labels in trainloader :
        # model.train()  by default in training model
        optimizer.zero_grad()
        outputs = model.forward(images)  # forward propogation
        loss = criterion(outputs , labels) 
        loss.backward()  # backward propogation
        optimizer.step() #update param

        epoch_training_loss += loss.item() #loss of 1 epoch
        
    print(f"epoch={epoch+1}/{epochs} and loss = {epoch_training_loss/len(trainloader)}")

epoch=1/10 and loss = 1.354639536431988
epoch=2/10 and loss = 0.9196157170378644
epoch=3/10 and loss = 0.7392596555564105
epoch=4/10 and loss = 0.6121607164440253
epoch=5/10 and loss = 0.5041975246747131
epoch=6/10 and loss = 0.4178143806202942
epoch=7/10 and loss = 0.33069046424782794
epoch=8/10 and loss = 0.25591426709538223
epoch=9/10 and loss = 0.19806487990729035
epoch=10/10 and loss = 0.15333809646899285


In [11]:
# # with validation

# epochs = 10 
# for epoch in range(epochs) :
#     epoch_training_loss = 0.0 
#     epoch_validation_loss = 0.0 

#     # TRAINING
#     for images , labels in trainloader :
#         # model.train()  by default in training model
#         optimizer.zero_grad()
#         outputs = model.forward(images)  # forward propogation
#         loss = criterion(outputs , labels) 
#         loss.backward()  # backward propogation
#         optimizer.step() #update param

#         epoch_training_loss += loss.item() #loss of 1 epoch

#     # VALIDATION
#     model.eval()  
    
#     with torch.no_grad():
#         for images , labels in testloader : 
            
#             outputs = model.forward(images)  # forward propogation
#             loss = criterion(outputs , labels) 
    
#             epoch_validation_loss += loss
        
#     print(f"epoch={epoch+1}/{epochs} , training loss = {epoch_training_loss/len(trainloader)} , validation loss  = {epoch_validation_loss/len(testloader)} ")

### Evaluate model

In [12]:
correct_labels = 0 
total_labels = 0 

model.eval() 

with torch.no_grad():
    for images, labels in testloader:

        outputs = model.forward(images)
        max , predicted = torch.max(outputs , 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print("Accuracy: " , correct_labels/total_labels *100)

Accuracy:  74.42999999999999
